# Stage 1 + the start of Stage 2 — Train the probe + `L_group`, tested on deliberately thinned cells

First implementation of the `L_group` formula (`notes/research_question/03_pivot2_group_consistency.md`
§12.13), tested in a Stage 2-style sparsification scenario (§12.7). This is the first pilot
that genuinely **trains a model** (not just extraction & measurement like notebooks 06/07).

**What is tested:** 3 `RACExRELIG` cells that originally have plenty of data (`White |
Protestant`, `Black | Protestant`, `White | Roman Catholic`) have their data deliberately
"thinned out" (really subsampled down to N=5/10/20/50 actual respondents),
and then 3 ways of guessing their real answers are compared:

1. **Plain probe** -- just the ordinary KL loss, without `L_group`.
2. **Probe + `L_group`** -- KL loss + the "rubber band" term that pulls the prediction
   towards the neighbouring cells whose representations are close (§12.13).
3. **Partial-pooling/shrinkage baseline** -- a simple version of the MRP logic
   (pull towards the neighbour average, with the pull strength depending on N),
   with no LLM at all.

If `L_group` really is useful, it has to be **more accurate than the plain probe**,
especially at small N -- and ideally comparable to or better than the shrinkage
baseline (which needs no LLM whatsoever, and is therefore the barometer for "does the
LLM add value or not", exactly the research question from
`research_question/00_overview.md` §9).

## Before running: Kaggle settings

1. **Accelerator**: GPU T4 x2 or P100. **Internet: On**.
2. **Upload 2 Kaggle Datasets**:
   - `opinionqa_intersectional.csv` (original path:
     `datasets/subpop/data/opinionqa/processed/opinionqa_intersectional.csv`)
     -- the same as for notebook 07.
   - **Raw data for 5 survey waves**: the folders
     `datasets/opinionqa_original/data/human_resp/American_Trends_Panel_W{42,49,92,45,50}/`
     (each folder contains `responses.csv` + `info.csv`). Upload all 5 of these folders
     as 1 Kaggle Dataset (you may zip them first, or upload all the files at once
     into 1 dataset, as long as the structure
     `.../American_Trends_Panel_W<number>/responses.csv` can still be found).

Estimated time: **60-90 minutes** (~15-20 minutes of representation extraction +
~40-50 minutes of probe training). The previous run hit OOM during extraction (the 7B
model turned out to be packed onto a single GPU by `device_map="auto"`) -- that is now
fixed (the model is forced to split across both T4s + the extraction batch is smaller +
there is an auto-retry if it still OOMs), but if extraction ends up slower than
estimated because of the small batch, that is expected (safety was prioritised over
speed after yesterday's OOM).

**Check the model-loading cell** (section 4): if it prints a lot of GPU memory still
free (>4 GB per GPU), `EXTRACTION_BATCH_SIZE` in the config cell can be raised manually
to make extraction faster.

**Check the diagnostic cell (section 6b) before waiting for the full run to finish**:
if the `KL=...` printed there every few epochs keeps going **up** (rather than down),
STOP -- something is wrong again, do not wait until the end. If it drops steadily, it is
safe to continue on to the full experiment below.

In [ ]:
!pip install -q -U "transformers>=4.44" accelerate scikit-learn tqdm
!pip install -q -U bitsandbytes

In [ ]:
import os
# Reduce CUDA memory fragmentation -- directly suggested by the OOM error of the
# previous run. It must be set BEFORE torch creates a CUDA context (hence on the
# very first line, before importing torch).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import ast
import gc
import sys
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import pairwise_distances
from scipy.stats import wasserstein_distance
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# IMPORTANT if this notebook has previously OOM-crashed in the SAME kernel
# (re-running cells without restarting the session): Jupyter/IPython automatically
# keeps the ENTIRE traceback of the last error via sys.last_traceback -- including
# every GPU tensor in every function frame that was running at the moment of the
# crash. That means the GPU memory from the earlier crash is NEVER freed even though
# the model itself looks like it has "been reloaded from scratch". The line below
# is only a precaution (cleaning up what CAN be cleaned up in the currently running
# kernel) -- if you really did just have an OOM crash, the safest and surest route is
# still to RESTART the Kaggle SESSION first (not merely re-run the cell), and only
# then Run All again from the top.
sys.last_traceback = None
gc.collect()

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        free_b, total_b = torch.cuda.mem_get_info(i)
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} -- {free_b / 1e9:.2f} GB free "
              f"of {total_b / 1e9:.2f} GB total")
        if free_b / total_b < 0.9:
            print(
                f"    !! GPU {i} is already significantly in use BEFORE the model is loaded. "
                "If this happens when you have only just started, it is very likely this "
                "kernel is left over from an earlier OOM crash -- RESTART the Kaggle SESSION "
                "first, and only then Run All again from the top."
            )
else:
    raise RuntimeError(
        "No GPU detected. Check Notebook options -> Accelerator -> GPU T4 x2/P100, "
        "then restart & Run All again."
    )

In [ ]:
# ── Model ────────────────────────────────────────────────────────
MODEL_PATH = "mistralai/Mistral-7B-v0.1"
USE_4BIT = False
# Lowered from 16 -- the previous run hit OOM because device_map="auto" turned out to
# pack the WHOLE 7B model (fp16 ~14GB) onto a single T4 (leaving ~100MB), with the
# second GPU completely idle. The model is now forced to split across BOTH T4s (see
# the model-loading cell below), so there is more room -- but we still start out
# conservatively; it can be raised again if the model-loading cell shows plenty of
# memory left (it prints that there). There is also an auto-retry that halves the
# batch if it still OOMs (see the extraction function).
EXTRACTION_BATCH_SIZE = 4

# Back to the CPU (as in the first run) for probe training -- tested locally, the GPU
# only gives about a 5% speed-up on this part (it is not compute-bound, but lots of
# small sequential operations per question), so it is not worth the risk of sharing
# GPU memory with the 7B model extraction, which needs that memory far more. The only
# thing that MUST be on the GPU is representation extraction (a 7B model, which makes
# no sense to run on the CPU -- it could take hours).
PROBE_DEVICE = "cpu"

# ── Find the data ────────────────────────────────────────────────
_candidates = glob.glob("/kaggle/input/**/opinionqa_intersectional.csv", recursive=True)
if _candidates:
    DATA_PATH = _candidates[0]
elif os.path.exists("opinionqa_intersectional.csv"):
    DATA_PATH = "opinionqa_intersectional.csv"
else:
    raise FileNotFoundError(
        "Could not find opinionqa_intersectional.csv. Upload it as a Kaggle Dataset first."
    )
print("Intersectional cell data from:", DATA_PATH)

RESTRICT_WAVES = [42, 49, 92, 45, 50]
WAVE_DIRS = {}
for w in RESTRICT_WAVES:
    cands = glob.glob(f"/kaggle/input/**/American_Trends_Panel_W{w}", recursive=True)
    if not cands:
        # try looking directly for its responses.csv file
        cands2 = glob.glob(f"/kaggle/input/**/American_Trends_Panel_W{w}/responses.csv", recursive=True)
        if cands2:
            cands = [os.path.dirname(cands2[0])]
    if not cands:
        raise FileNotFoundError(
            f"Could not find the folder American_Trends_Panel_W{w} in /kaggle/input. "
            "Upload the raw data for these 5 waves as a Kaggle Dataset first."
        )
    WAVE_DIRS[w] = cands[0]
print("Raw wave data found at:")
for w, d in WAVE_DIRS.items():
    print(f"  W{w}: {d}")

# ── Experiment scenario ───────────────────────────────────────────
ATTR_TYPE = "RACExRELIG"
SPARSE_CELLS = ["White | Protestant", "Black | Protestant", "White | Roman Catholic"]
N_LEVELS = [5, 10, 20, 50]
N_REPEATS = 3  # lowered from 5 -- the epoch count went up a lot, this keeps the total runtime sane

LAYER_START, LAYER_END = 17, 27  # inclusive, as per §12.13.3
K_NEIGHBORS = 5
LAMBDA_GROUP = 1.0
N_EPOCHS = 300  # raised from 30 -- in the first run the probe looked like it never got a chance to learn
LEARNING_RATE = 0.001  # lowered from 0.01 -- in the first run the loss DIVERGED (kept rising),
                       # not merely converged slowly. Raw LLM representations are large in scale
                       # (not small 0-1 numbers), and lr=0.01 is too big for that. Checked locally:
                       # lr=0.01 diverges at large scales, lr=0.001 is stable at every scale tried.

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

OUT_DIR = "/kaggle/working/stage1_lgroup_pilot"
assert not OUT_DIR.startswith("/kaggle/input"), "OUT_DIR must be under /kaggle/working!"
os.makedirs(OUT_DIR, exist_ok=True)
print("\nOUT_DIR:", OUT_DIR)
print("PROBE_DEVICE:", PROBE_DEVICE)

## 1. Prepare the data: `RACExRELIG` cells, K=2, from the 5 selected waves

In [ ]:
df = pd.read_csv(DATA_PATH)

def _parse(x):
    return ast.literal_eval(x) if isinstance(x, str) else x

df["responses"] = df["responses"].apply(_parse)
df["ordinal"] = df["ordinal"].apply(_parse)
df["options"] = df["options"].apply(_parse)

sub = df[
    (df["attribute"] == ATTR_TYPE)
    & (df["ordinal"].apply(len) == 2)
    & (df["wave"].isin(RESTRICT_WAVES))
].reset_index(drop=True)

GROUP_KEYS = sorted(sub["group"].unique().tolist())
n_g = len(GROUP_KEYS)
print(f"Total rows: {len(sub)}")
print(f"Number of cells: {n_g}")
print(f"Number of qkeys (K=2, the 5 selected waves): {sub['qkey'].nunique()}")
for c in SPARSE_CELLS:
    assert c in GROUP_KEYS, f"Sparsify cell {c} is not present in the filtered data!"
print("All target sparsify cells are present in the data. OK.")

# fast lookup: (qkey, group) -> row
row_lookup = sub.set_index(["qkey", "group"]).to_dict("index")
qkeys_by_cell = sub.groupby("group")["qkey"].apply(set).to_dict()
QKEYS = sorted(sub["qkey"].unique().tolist())

## 2. Build the "deliberately little data" version for the 3 target cells

For each (qkey, target cell), pull the ORIGINAL respondents back out of the raw wave
file, subsample down to N=5/10/20/50 (5 repetitions per N so that one odd sample does
not decide things), and compute the answer distribution from that small sample -- this
is what the model is trained on (not the real distribution).

The REAL distribution (from the full sample, the `responses` column in the data) is
kept as the answer key for measuring accuracy later.

In [ ]:
wave_raw = {w: pd.read_csv(os.path.join(d, "responses.csv"), low_memory=False) for w, d in WAVE_DIRS.items()}
print("Raw waves loaded:", {w: len(df_) for w, df_ in wave_raw.items()})

def subsample_response(cell, qkey, n, seed):
    row = row_lookup[(qkey, cell)]
    wave = row["wave"]
    ordinal_refs = row["options"][: len(row["ordinal"])]
    race, relig = cell.split(" | ")
    weight_col = f"WEIGHT_W{wave}"
    wdf = wave_raw[wave]
    pool = wdf[(wdf["RACE"] == race) & (wdf["RELIG"] == relig) & (wdf[qkey].isin(ordinal_refs))]
    if len(pool) == 0:
        return None
    local_rng = np.random.default_rng(seed)
    idx = local_rng.choice(len(pool), size=min(n, len(pool)), replace=False)
    draw = pool.iloc[idx]
    counts = {ref: draw.loc[draw[qkey] == ref, weight_col].sum() for ref in ordinal_refs}
    total = sum(counts.values())
    if total <= 0:
        return None
    return [counts[r] / total for r in ordinal_refs]

# build the sparsification labels: (cell, qkey, n, repeat) -> small-sample distribution
sparse_labels = {}
skipped = 0
for cell in SPARSE_CELLS:
    for qkey in tqdm(sorted(qkeys_by_cell[cell]), desc=f"Sparsifying {cell}"):
        for n in N_LEVELS:
            for rep in range(N_REPEATS):
                seed = hash((cell, qkey, n, rep)) % (2**31)
                dist = subsample_response(cell, qkey, n, seed)
                if dist is None:
                    skipped += 1
                    continue
                sparse_labels[(cell, qkey, n, rep)] = dist

print(f"\nTotal sparsification labels created: {len(sparse_labels)} (skipped: {skipped})")

## 3. Build the prompts: group (for the `w_ij` graph) & combined question+group (probe input)

In [ ]:
def build_group_prompt(cell):
    race, relig = cell.split(" | ")
    return f"This survey respondent's race is {race} and their religion is {relig}."

def build_combined_prompt(cell, qkey):
    row = row_lookup[(qkey, cell)]
    q, opts = row["question"], row["options"]
    letters = [chr(ord("A") + i) for i in range(len(opts))]
    lines = [build_group_prompt(cell), "", q] + [f"{l}. {o}" for l, o in zip(letters, opts)] + ["Answer:"]
    return "\n".join(lines)

print("Example group prompt:\n", build_group_prompt(GROUP_KEYS[0]))
print("\nExample combined prompt:\n", build_combined_prompt(GROUP_KEYS[0], sorted(qkeys_by_cell[GROUP_KEYS[0]])[0]))

## 4. Load the model, extract representations (average of layers 17-27)

In [ ]:
print(f"Loading tokenizer & model: {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # important: keeps the LAST token position identical even when batched

# device_map="balanced" (NOT "auto") -- forces a split across ALL available GPUs.
# The previous run hit OOM because "auto" turned out to put the WHOLE 7B model (fp16
# ~14GB) on a single T4 (leaving ~100MB free), while the second T4 sat completely idle
# -- that is the root cause of the GPU being "underused" from the start. "balanced"
# forces accelerate to spread the layers evenly over both GPUs even though the model
# itself would fit on 1 GPU, so that there is free room for the extraction batch activations.
model_kwargs = dict(torch_dtype=torch.float16, device_map="balanced", low_cpu_mem_usage=True)
if USE_4BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs.pop("torch_dtype", None)
    model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, **model_kwargs)
model.eval()
NUM_LAYERS = model.config.num_hidden_layers
print(f"Model loaded. Number of layers: {NUM_LAYERS}")
print("Model spread across GPUs (hf_device_map):", getattr(model, "hf_device_map", "?"))
assert LAYER_END < NUM_LAYERS, "LAYER_END exceeds the number of layers in the model!"

for i in range(torch.cuda.device_count()):
    free_b, total_b = torch.cuda.mem_get_info(i)
    print(f"  GPU {i}: {free_b / 1e9:.2f} GB free of {total_b / 1e9:.2f} GB "
          f"(after the model was loaded) -- if this is still plenty (>4 GB), "
          f"EXTRACTION_BATCH_SIZE in the config cell can be raised again.")


@torch.no_grad()
def get_avg_layer_embeddings_batch(prompts, batch_size=EXTRACTION_BATCH_SIZE):
    """Average of the LAST token's representation over layers LAYER_START..LAYER_END,
    processed per batch (rather than one at a time) so the GPU is used more fully.

    Uses left-padding + explicit position_ids (not the default arange), so that the
    result is identical to processing one at a time -- validated locally with a small
    model (hf-internal-testing/tiny-random-gpt2); the maximum difference between the
    batched and the one-at-a-time version was on the order of 1e-6 (just floating
    point rounding, not a bug).

    If this batch_size is still too large for the available memory, catch
    OutOfMemoryError and automatically retry with half the batch -- so you do not have
    to guess exactly the right number up front / fail outright in the middle of a
    process that has already been running a long time.
    """
    all_embeddings = []
    i = 0
    bs = batch_size
    while i < len(prompts):
        batch = prompts[i : i + bs]
        try:
            enc = tokenizer(batch, return_tensors="pt", padding=True).to(model.device)
            attn = enc["attention_mask"]
            position_ids = (attn.cumsum(-1) - 1).masked_fill(attn == 0, 1)

            out = model(
                input_ids=enc["input_ids"],
                attention_mask=attn,
                position_ids=position_ids,
                output_hidden_states=True,
            )
            # slice to the layer window we use FIRST, then stack -- this uses less
            # memory than stacking all 33 layers and only then slicing.
            hs_window = torch.stack(out.hidden_states[LAYER_START : LAYER_END + 1], dim=0)
            last_token = hs_window[:, :, -1, :]  # left-padded -> position -1 is always the real last token
            avg = last_token.mean(dim=0)  # (batch, hidden)
            all_embeddings.append(avg.float().cpu().numpy())

            del out, hs_window, last_token, avg, enc, attn, position_ids
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            i += bs
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs <= 1:
                raise
            bs = max(1, bs // 2)
            print(f"  [OOM] dropping the batch to {bs} and retrying from prompt #{i}...")

    return np.concatenate(all_embeddings, axis=0)

In [ ]:
# group-only representations (for the w_ij graph) -- processed per batch
cell_prompts_list = [build_group_prompt(c) for c in GROUP_KEYS]
cell_emb_array = get_avg_layer_embeddings_batch(cell_prompts_list)
cell_embeddings = {c: cell_emb_array[i] for i, c in enumerate(GROUP_KEYS)}
print("Shape of the group representations:", cell_emb_array.shape)

In [ ]:
# combined question+group representations (probe input) -- processed per batch,
# once per valid (qkey, cell)
valid_pairs = [(qkey, cell) for qkey in QKEYS for cell in GROUP_KEYS if (qkey, cell) in row_lookup]
print(f"Number of valid (qkey, cell) pairs: {len(valid_pairs)}")

pair_prompts_list = [build_combined_prompt(cell, qkey) for qkey, cell in valid_pairs]
pair_emb_array_raw = get_avg_layer_embeddings_batch(pair_prompts_list)
pair_embeddings = {pair: pair_emb_array_raw[i] for i, pair in enumerate(valid_pairs)}

# Normalise to vector length 1 (L2 norm) -- raw LLM representations are large in scale
# and inconsistent, and feeding them straight into the probe can make training unstable
# (which is what happened in the first run: the loss went up/diverged instead of down).
# This is an EXTRA safeguard on top of lowering LEARNING_RATE in the config cell -- so
# that we do not depend solely on guessing one exactly-right learning rate.
raw_norms = np.array([np.linalg.norm(v) for v in pair_embeddings.values()])
print(f"Representation vector lengths BEFORE normalisation: min={raw_norms.min():.2f}, "
      f"median={np.median(raw_norms):.2f}, max={raw_norms.max():.2f}")
pair_embeddings = {k: v / np.linalg.norm(v) for k, v in pair_embeddings.items()}

# ALSO save the combined representations (not only the group-only ones) -- so that if we
# need to debug training again later, we do not have to redo the extraction on the GPU.
pair_qkeys = np.array([q for q, c in valid_pairs], dtype=object)
pair_cells = np.array([c for q, c in valid_pairs], dtype=object)
pair_emb_array = np.stack([pair_embeddings[(q, c)] for q, c in valid_pairs])

## 5. Build the `w_ij` graph (k-NN + Gaussian kernel, median heuristic)

Because every cell here is of one combination type (`RACExRELIG`), the "different type
= zero" gate from §12.13.3 is automatically never active -- all pairs are allowed into
the kernel, restricted only via the k nearest neighbours.

In [ ]:
def build_knn_graph(emb_array, k):
    norm = emb_array / np.linalg.norm(emb_array, axis=1, keepdims=True)
    cos_dist = 1 - (norm @ norm.T)
    np.fill_diagonal(cos_dist, np.inf)
    sigma = np.median(cos_dist[np.isfinite(cos_dist)])
    w = np.exp(-(cos_dist ** 2) / (2 * sigma ** 2))
    n = emb_array.shape[0]
    knn_mask = np.zeros((n, n), dtype=bool)
    for i in range(n):
        nn_idx = np.argsort(cos_dist[i])[:k]
        knn_mask[i, nn_idx] = True
    knn_mask = knn_mask | knn_mask.T
    w = np.where(knn_mask, w, 0.0)
    np.fill_diagonal(w, 0.0)
    return w, sigma

W_IJ, SIGMA = build_knn_graph(cell_emb_array, K_NEIGHBORS)
print(f"Sigma (median heuristic): {SIGMA:.4f}")
print(f"Number of non-zero edges: {(W_IJ > 0).sum()} out of {n_g * (n_g - 1)} possible")

for cell in SPARSE_CELLS:
    i = GROUP_KEYS.index(cell)
    neighbors = [(GROUP_KEYS[j], W_IJ[i, j]) for j in np.argsort(-W_IJ[i]) if W_IJ[i, j] > 0][:5]
    print(f"\nNearest neighbours of '{cell}': {neighbors}")

## 6. The probe, the `L_group` loss (JSD), and the training loop

In [ ]:
def jsd_matrix(pred, eps=1e-8):
    """Jensen-Shannon Divergence between ALL pairs at once (as a matrix), rather than
    looping one by one -- much faster. pred: (n, K) -> result (n, n)."""
    p_i = pred.unsqueeze(1).clamp(min=eps)  # (n, 1, K)
    p_j = pred.unsqueeze(0).clamp(min=eps)  # (1, n, K)
    m = 0.5 * (p_i + p_j)
    kl_i = (p_i * (p_i / m).log()).sum(dim=-1)  # (n, n)
    kl_j = (p_j * (p_j / m).log()).sum(dim=-1)  # (n, n)
    return 0.5 * kl_i + 0.5 * kl_j


# All the combined representations are moved into a single GPU tensor once here (rather
# than a numpy dict re-converted every epoch) -- this is what makes the training loop
# genuinely use the GPU instead of the CPU as before (an idle GPU during those ~40-50
# minutes of training was the main cause of "the GPU is underused").
pair_index = {pair: i for i, pair in enumerate(valid_pairs)}
pair_emb_tensor = torch.tensor(pair_emb_array, dtype=torch.float32, device=PROBE_DEVICE)
W_IJ_full = torch.tensor(W_IJ, dtype=torch.float32, device=PROBE_DEVICE)
GROUP_INDEX = {c: i for i, c in enumerate(GROUP_KEYS)}


def train_probe(qkey_to_cell_labels, use_group_loss, seed=RANDOM_SEED, n_epochs=N_EPOCHS,
                 lr=LEARNING_RATE, verbose=False, log_every=None):
    """qkey_to_cell_labels: dict qkey -> {cell: target distribution (list of length 2)}.
    Trains 1 linear probe (D->2) over all qkeys at once, on the GPU (PROBE_DEVICE).
    Returns: (probe, history) -- history = list of (mean KL, mean L_group) per epoch,
    used to check whether training really converged (see the diagnostic cell)."""
    torch.manual_seed(seed)
    D = pair_emb_tensor.shape[1]
    probe = nn.Linear(D, 2).to(PROBE_DEVICE)
    optimizer = torch.optim.Adam(probe.parameters(), lr=lr)

    # pre-compute the neighbour indices/labels/weights ONCE per qkey (rather than
    # repeating it every epoch) -- stored directly as GPU tensors so the epoch loop is
    # purely GPU operations with no CPU<->GPU round trips.
    qkey_data = {}
    for qkey, cell_labels in qkey_to_cell_labels.items():
        if len(cell_labels) < 2:
            continue
        cells_present = list(cell_labels.keys())
        idx_tensor = torch.tensor([pair_index[(qkey, c)] for c in cells_present], device=PROBE_DEVICE)
        Y = torch.tensor(np.stack([cell_labels[c] for c in cells_present]), dtype=torch.float32, device=PROBE_DEVICE)
        group_idx = [GROUP_INDEX[c] for c in cells_present]
        w_sub = W_IJ_full[group_idx][:, group_idx]
        qkey_data[qkey] = (idx_tensor, Y, w_sub)

    history = []
    for epoch in range(n_epochs):
        epoch_kl, epoch_lg = [], []
        for idx_tensor, Y, w_sub in qkey_data.values():
            X = pair_emb_tensor[idx_tensor]  # (n, D), already on the GPU, just index it

            logits = probe(X)
            log_probs = torch.log_softmax(logits, dim=-1)
            pred = torch.softmax(logits, dim=-1)

            kl = (Y * (Y.clamp(min=1e-8).log() - log_probs)).sum(dim=-1).mean()
            loss = kl
            lg_value = 0.0

            if use_group_loss:
                total_w = w_sub.sum()
                if total_w > 0:
                    jm = jsd_matrix(pred)
                    l_group_term = (w_sub * jm).sum() / total_w
                    loss = loss + LAMBDA_GROUP * l_group_term
                    lg_value = l_group_term.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_kl.append(kl.item())
            epoch_lg.append(lg_value)

        history.append((float(np.mean(epoch_kl)), float(np.mean(epoch_lg))))
        if verbose and log_every and (epoch % log_every == 0 or epoch == n_epochs - 1):
            print(f"    epoch {epoch:4d}: KL={history[-1][0]:.4f}  L_group={history[-1][1]:.4f}")

    return probe, history


@torch.no_grad()
def predict_probe(probe, qkey, cell):
    idx = pair_index[(qkey, cell)]
    X = pair_emb_tensor[idx].unsqueeze(0)
    return torch.softmax(probe(X), dim=-1).squeeze(0).cpu().numpy()

# The shrinkage baseline (simple partial pooling) is defined in the next cell, after
# true_labels (used as the "prior" from the neighbours) has been prepared.

## 6b. Diagnostics first, before the full experiment

In the first run of this pilot the probe was terrible (WD 0.40-0.48, worse than random
guessing). Before repeating the full experiment (expensive, ~30-50 minutes), check 2
cheap things first:

1. **Does the loss actually go down** when trained for longer (300 epochs rather than
   30) -- printed every few epochs below.
2. **Can the probe "connect" to the data it SEES DIRECTLY** during training (rather
   than to the sparsified cells) -- if it is still bad even there, then training itself
   is not right yet (it is not a question of generalisation, let alone of `L_group`).

In [ ]:
# true_labels: (qkey, cell) -> the real distribution (from the data, full N) -- used
# both in these diagnostics and in the main experiment later.
true_labels = {(qkey, cell): row_lookup[(qkey, cell)]["responses"] for (qkey, cell) in valid_pairs}

print("=== Diagnostics: train 1 example probe (N=50 scenario, WITHOUT L_group), watch the loss per epoch ===")
n_diag, rep_diag = 50, 0
qkey_to_cell_labels_diag = {}
for qkey in QKEYS:
    entry = {}
    for cell in GROUP_KEYS:
        if (qkey, cell) not in row_lookup:
            continue
        if cell in SPARSE_CELLS:
            key = (cell, qkey, n_diag, rep_diag)
            if key in sparse_labels:
                entry[cell] = sparse_labels[key]
        else:
            entry[cell] = true_labels[(qkey, cell)]
    if len(entry) >= 2:
        qkey_to_cell_labels_diag[qkey] = entry

probe_diag, history_diag = train_probe(
    qkey_to_cell_labels_diag, use_group_loss=False, seed=999,
    log_every=max(1, N_EPOCHS // 10), verbose=True,
)

print("\n=== Fit check: does the probe match the data IT ITSELF saw during training? ===")
normal_cells = [c for c in GROUP_KEYS if c not in SPARSE_CELLS]
sample_check = [
    (qkey, cell) for qkey in list(qkey_to_cell_labels_diag)[:20]
    for cell in normal_cells if (qkey, cell) in true_labels
][:40]
fit_wds = []
for qkey, cell in sample_check:
    pred = predict_probe(probe_diag, qkey, cell)
    ordinal = row_lookup[(qkey, cell)]["ordinal"]
    wd = wasserstein_distance(ordinal, ordinal, u_weights=np.clip(pred, 1e-8, None), v_weights=true_labels[(qkey, cell)])
    fit_wds.append(wd)
print(f"Mean WD over the {len(fit_wds)} examples IT TRAINED ON DIRECTLY: {np.mean(fit_wds):.4f}")
print("- If this is still high (>0.2): the probe has not 'connected' to its data -- raise N_EPOCHS again or lower LEARNING_RATE, and do NOT move on to the main experiment yet.")
print("- If this is already low: it is safe to continue to the main experiment below -- only then does generalisation to the sparsify cells (and L_group) become relevant to test.")

In [ ]:
from scipy.stats import wasserstein_distance

In [ ]:
# neighbor_prior/shrinkage_predict use the true_labels prepared in the diagnostic cell
# (6b) above -- the neighbours here use the neighbours' REAL labels (information that is
# genuinely available, since we only sparsified 3 target cells).
def neighbor_prior(qkey, cell):
    i = GROUP_KEYS.index(cell)
    neighbor_idx = np.where(W_IJ[i] > 0)[0]
    dists, weights = [], []
    for j in neighbor_idx:
        nb_cell = GROUP_KEYS[j]
        if (qkey, nb_cell) in true_labels:
            dists.append(true_labels[(qkey, nb_cell)])
            weights.append(W_IJ[i, j])
    if not dists:
        return None
    dists = np.array(dists)
    weights = np.array(weights)
    return (dists * weights[:, None]).sum(axis=0) / weights.sum()

def shrinkage_predict(sparse_dist, n_obs, qkey, cell, tau=20.0):
    prior = neighbor_prior(qkey, cell)
    if prior is None:
        return np.array(sparse_dist)
    alpha = n_obs / (n_obs + tau)
    return alpha * np.array(sparse_dist) + (1 - alpha) * prior


results = []
for n in N_LEVELS:
    for rep in range(N_REPEATS):
        # build the training labels for this (n, rep) scenario
        qkey_to_cell_labels = {}
        for qkey in QKEYS:
            entry = {}
            for cell in GROUP_KEYS:
                if (qkey, cell) not in row_lookup:
                    continue
                if cell in SPARSE_CELLS:
                    key = (cell, qkey, n, rep)
                    if key in sparse_labels:
                        entry[cell] = sparse_labels[key]
                    # if there is no valid draw, skip this cell for this qkey
                else:
                    entry[cell] = true_labels[(qkey, cell)]
            if len(entry) >= 2:
                qkey_to_cell_labels[qkey] = entry

        probe_plain, hist_plain = train_probe(qkey_to_cell_labels, use_group_loss=False, seed=1000 + rep)
        probe_group, hist_group = train_probe(qkey_to_cell_labels, use_group_loss=True, seed=1000 + rep)
        print(
            f"N={n} repeat={rep+1}/{N_REPEATS}: "
            f"plain KL {hist_plain[0][0]:.3f}->{hist_plain[-1][0]:.3f}, "
            f"KL+L_group {hist_group[0][0]:.3f}->{hist_group[-1][0]:.3f} "
            f"(final L_group={hist_group[-1][1]:.3f})"
        )

        for cell in SPARSE_CELLS:
            for qkey in sorted(qkeys_by_cell[cell]):
                key = (cell, qkey, n, rep)
                if key not in sparse_labels or qkey not in qkey_to_cell_labels or cell not in qkey_to_cell_labels[qkey]:
                    continue
                true_dist = true_labels[(qkey, cell)]
                ordinal = row_lookup[(qkey, cell)]["ordinal"]

                pred_plain = predict_probe(probe_plain, qkey, cell)
                pred_group = predict_probe(probe_group, qkey, cell)
                pred_shrink = shrinkage_predict(sparse_labels[key], n, qkey, cell)

                for method, pred in [("probe_plain", pred_plain), ("probe_L_group", pred_group), ("shrinkage", pred_shrink)]:
                    wd = wasserstein_distance(ordinal, ordinal, u_weights=np.clip(pred, 1e-8, None), v_weights=true_dist)
                    results.append({"n": n, "repeat": rep, "cell": cell, "qkey": qkey, "method": method, "wd": wd})

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(OUT_DIR, "pilot_results_raw.csv"), index=False)
print(f"\nTotal result rows: {len(results_df)}")

## 8. Summarise & plot: WD vs N, per method

This is the AutoElicit-style curve (§8.2) -- if `L_group` really helps, its line has to
sit below `probe_plain`, especially at small N.

In [ ]:
summary = results_df.groupby(["n", "method"])["wd"].agg(["mean", "std", "count"]).reset_index()
print(summary.to_string(index=False))
summary.to_csv(os.path.join(OUT_DIR, "pilot_summary.csv"), index=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = {"probe_plain": "#d62728", "probe_L_group": "#1f77b4", "shrinkage": "#2ca02c"}
for method in ["probe_plain", "probe_L_group", "shrinkage"]:
    sub_s = summary[summary["method"] == method].sort_values("n")
    ax.errorbar(sub_s["n"], sub_s["mean"], yerr=sub_s["std"], marker="o", label=method, color=colors[method], capsize=3)
ax.set_xlabel("N (number of respondents simulated as available)")
ax.set_ylabel("Wasserstein Distance to the real answers (smaller is better)")
ax.set_title("Stage 1 pilot -- accuracy vs N, 3 methods")
ax.set_xscale("log")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "pilot_wd_vs_n.png"), dpi=150)
plt.show()

print(f"\nAll outputs are in: {OUT_DIR}")

## 9. Follow-up investigation: why do `L_group` & N appear to make no difference?

The table above shows 2 things that need digging into before concluding anything about
`L_group`:
1. `probe_L_group` is almost identical to `probe_plain` at EVERY level of N (the gap is
   smaller than reasonable random error) -- `L_group` looks like it is not "working".
2. Neither probe (plain or +`L_group`) improves as N (the amount of real data pretended
   to be available) rises from 5 to 50 -- whereas `shrinkage` (simple statistics, no
   LLM) keeps improving as N grows.

The 5 checks below (cheap, no GPU re-extraction needed -- just retraining a few small
probes on the representations we already have) are designed to work out whether this is
a matter of experiment DESIGN (the probe not "hearing" its own sparse cell labels) or
whether `L_group` really is weak:

- **A+B**: retrain 2 small probes (the N=5 vs N=50 scenarios, ~3-5 minutes) -- compare
  their predictions DIRECTLY for the same cell & question, and check how closely the
  probe fits its OWN SPARSE LABELS (not just the real/true labels used for the final
  score).
- **C+E**: check how much `w_ij` graph weight is connected to the 3 target cells, and
  whether their nearest neighbours (according to the LLM representations) actually make
  sense in terms of the real answers (rather than just being "close" but unrelated).
- **D**: force `LAMBDA_GROUP` up to 100x its original value (1.0) -- if even that
  changes nothing, then `L_group` is structurally without effect here (not merely a
  matter of a badly chosen lambda value).
- **F+G**: check where the embeddings sit GEOMETRICALLY -- are all the cells really
  spread out clearly in representation space, or do they cluster into a single blob
  (making the kernel less discriminative)? And most directly of all: use a **RANDOM**
  probe (not trained at all) to measure `L_group` -- if even a random probe already
  yields a small JSD to the neighbours, that is evidence that the geometry of the
  representations itself is what makes them "close", NOT `L_group` training pulling
  them there.

In [ ]:
print("=" * 70)
print("INVESTIGATION A+B: are the probe's predictions sensitive to N? does the probe")
print("fit its OWN sparse labels?")
print("=" * 70)

def build_qkey_to_cell_labels(n, rep):
    out = {}
    for qkey in QKEYS:
        entry = {}
        for cell in GROUP_KEYS:
            if (qkey, cell) not in row_lookup:
                continue
            if cell in SPARSE_CELLS:
                key = (cell, qkey, n, rep)
                if key in sparse_labels:
                    entry[cell] = sparse_labels[key]
            else:
                entry[cell] = true_labels[(qkey, cell)]
        if len(entry) >= 2:
            out[qkey] = entry
    return out

diag_probes = {}
for n in [5, 50]:
    qtl = build_qkey_to_cell_labels(n, 0)
    probe_n, _ = train_probe(qtl, use_group_loss=False, seed=2000)
    diag_probes[n] = (probe_n, qtl)
    print(f"Finished training the diagnostic probe N={n} (rep=0, without L_group)")

# ── A: compare the predictions & training labels DIRECTLY at N=5 vs N=50 ──
print(f"\n--- A. Concrete example: cell '{SPARSE_CELLS[0]}', first 5 questions ---")
sample_qkeys = sorted(qkeys_by_cell[SPARSE_CELLS[0]])[:5]
for qkey in sample_qkeys:
    cell = SPARSE_CELLS[0]
    key5, key50 = (cell, qkey, 5, 0), (cell, qkey, 50, 0)
    if key5 not in sparse_labels or key50 not in sparse_labels:
        continue
    pred5 = predict_probe(diag_probes[5][0], qkey, cell)
    pred50 = predict_probe(diag_probes[50][0], qkey, cell)
    print(f"  {qkey}:")
    print(f"    training label @N=5  = {np.round(sparse_labels[key5], 2)}   -> probe prediction = {np.round(pred5, 2)}")
    print(f"    training label @N=50 = {np.round(sparse_labels[key50], 2)}   -> probe prediction = {np.round(pred50, 2)}")
    print(f"    (the REAL/true answer = {np.round(true_labels[(qkey, cell)], 2)})")

pred_diffs, label_diffs = [], []
for cell in SPARSE_CELLS:
    for qkey in sorted(qkeys_by_cell[cell]):
        key5, key50 = (cell, qkey, 5, 0), (cell, qkey, 50, 0)
        if key5 not in sparse_labels or key50 not in sparse_labels:
            continue
        if qkey not in diag_probes[5][1] or cell not in diag_probes[5][1][qkey]:
            continue
        pred5 = predict_probe(diag_probes[5][0], qkey, cell)
        pred50 = predict_probe(diag_probes[50][0], qkey, cell)
        pred_diffs.append(np.abs(pred5 - pred50).sum())
        label_diffs.append(np.abs(np.array(sparse_labels[key5]) - np.array(sparse_labels[key50])).sum())

print(f"\nOver {len(pred_diffs)} (qkey, target cell) pairs:")
print(f"  Mean difference in the TRAINING LABELS (given to the probe, N=5 vs N=50): {np.mean(label_diffs):.4f}")
print(f"  Mean difference in the resulting probe PREDICTIONS   (N=5 vs N=50): {np.mean(pred_diffs):.4f}")
print("  -> if the PREDICTION difference is far SMALLER than the LABEL difference, it means")
print("     the probe is NOT 'listening' much to its own sparse cell labels --")
print("     its predictions are dominated by the pattern from the other 23 cells whose labels are stable.")

# ── B: how closely does the probe fit its OWN sparse labels (not true) ──
print(f"\n--- B. How closely the probe fits its OWN SPARSE labels (not true) ---")
for n in [5, 50]:
    probe_n, qtl_n = diag_probes[n]
    own_wds = []
    for cell in SPARSE_CELLS:
        for qkey in sorted(qkeys_by_cell[cell]):
            key = (cell, qkey, n, 0)
            if key not in sparse_labels or qkey not in qtl_n or cell not in qtl_n[qkey]:
                continue
            pred = predict_probe(probe_n, qkey, cell)
            ordinal = row_lookup[(qkey, cell)]["ordinal"]
            wd_own = wasserstein_distance(
                ordinal, ordinal, u_weights=np.clip(pred, 1e-8, None), v_weights=sparse_labels[key]
            )
            own_wds.append(wd_own)
    print(f"  N={n}: WD to its OWN sparse labels = {np.mean(own_wds):.4f} (n={len(own_wds)} pairs)")
print("  (compare with the training-fit WD of the NORMAL cells in diagnostic 6b, around ~0.12 --")
print("   if the number for these 3 target cells is far HIGHER than that, it means the probe")
print("   really is losing the competition for the model's capacity against the other 23 cells.)")

In [ ]:
print("=" * 70)
print("INVESTIGATION C+E: how large & how sensible is the L_group 'pull' on these 3 cells")
print("=" * 70)

print("\n--- C. The w_ij weight connected to the 3 target cells (vs the average over all cells) ---")
avg_total_w = W_IJ.sum(axis=1).mean()
for cell in SPARSE_CELLS:
    i = GROUP_KEYS.index(cell)
    total_w = W_IJ[i].sum()
    n_edges = int((W_IJ[i] > 0).sum())
    flag = "  <-- below average!" if total_w < avg_total_w else ""
    print(f"  {cell}: total neighbour weight = {total_w:.4f} ({n_edges} neighbours){flag}")
print(f"  Average over ALL cells: {avg_total_w:.4f}")
print("  (if the weight of these 3 target cells is far below the average, it means the kernel")
print("   simply does not find neighbours that are 'close' for these cells -- L_group does not")
print("   have much material to pull them anywhere with.)")

print("\n--- E. Are the neighbours (according to the LLM representations) actually similar IN THE REAL ANSWERS? ---")
for cell in SPARSE_CELLS:
    i = GROUP_KEYS.index(cell)
    neighbor_idx = [j for j in np.argsort(-W_IJ[i]) if W_IJ[i, j] > 0][:3]
    sample_qkey = sorted(qkeys_by_cell[cell])[0]
    true_this = true_labels.get((sample_qkey, cell))
    print(f"\n  {cell} (example qkey={sample_qkey}):")
    print(f"    true dist THIS CELL    = {np.round(true_this, 3) if true_this is not None else 'N/A'}")
    for j in neighbor_idx:
        nb = GROUP_KEYS[j]
        true_nb = true_labels.get((sample_qkey, nb))
        diff = (
            np.abs(np.array(true_this) - np.array(true_nb)).sum()
            if (true_this is not None and true_nb is not None) else None
        )
        diff_str = f", difference={diff:.3f}" if diff is not None else ""
        print(f"    neighbour '{nb}' (w_ij={W_IJ[i, j]:.3f}): true dist = "
              f"{np.round(true_nb, 3) if true_nb is not None else 'N/A'}{diff_str}")
print("\n  (if the true-dist difference to the neighbours is in fact large, it means the LLM")
print("   representations consider these cells 'close' even though their real answers are far")
print("   apart -- L_group could then be pulling in the WRONG direction, not merely 'having no effect'.)")

In [ ]:
print("=" * 70)
print("INVESTIGATION D: if LAMBDA_GROUP is forced far up, does L_group start to matter?")
print("=" * 70)

qtl_test = diag_probes[5][1]  # the N=5 scenario -- the one that most needs L_group's help
_LAMBDA_BACKUP = LAMBDA_GROUP

for lam_test in [1.0, 10.0, 100.0]:
    LAMBDA_GROUP = lam_test
    probe_g, hist_g = train_probe(qtl_test, use_group_loss=True, seed=3000)
    wds = []
    for cell in SPARSE_CELLS:
        for qkey in sorted(qkeys_by_cell[cell]):
            key = (cell, qkey, 5, 0)
            if key not in sparse_labels or qkey not in qtl_test or cell not in qtl_test[qkey]:
                continue
            pred = predict_probe(probe_g, qkey, cell)
            ordinal = row_lookup[(qkey, cell)]["ordinal"]
            wd = wasserstein_distance(
                ordinal, ordinal, u_weights=np.clip(pred, 1e-8, None),
                v_weights=true_labels[(qkey, cell)],
            )
            wds.append(wd)
    print(f"  LAMBDA_GROUP={lam_test:>6}: final L_group={hist_g[-1][1]:.4f}, "
          f"final KL={hist_g[-1][0]:.4f}, mean WD to TRUE = {np.mean(wds):.4f}")

LAMBDA_GROUP = _LAMBDA_BACKUP
print(f"\n(LAMBDA_GROUP restored to {LAMBDA_GROUP} after this test)")
print("\n-> if the WD barely changes even with lambda raised 100x, then L_group is")
print("   STRUCTURALLY without effect here (not merely a matter of a badly chosen lambda")
print("   value) -- and that distinction matters: if the structure is what is wrong,")
print("   raising lambda in later experiments will not help.")
print("   If the WD actually gets WORSE as lambda rises, that is a sign that L_group is")
print("   pulling in the wrong direction (see investigation E above).")

In [ ]:
print("=" * 70)
print("INVESTIGATION F+G: where do the embeddings sit GEOMETRICALLY? are they")
print("'close' to begin with (in the representations), rather than by training?")
print("=" * 70)

# ── F: distance statistics (cosine distance) over ALL cell pairs ──
norm = cell_emb_array / np.linalg.norm(cell_emb_array, axis=1, keepdims=True)
cos_dist_all = 1 - (norm @ norm.T)
iu = np.triu_indices_from(cos_dist_all, k=1)
all_dists = cos_dist_all[iu]

print(f"\n--- F. Distance statistics over ALL {n_g} cells ({len(all_dists)} pairs) ---")
print(f"  min={all_dists.min():.5f}  median={np.median(all_dists):.5f}  "
      f"mean={all_dists.mean():.5f}  max={all_dists.max():.5f}  std={all_dists.std():.5f}")
print(f"  SIGMA (median heuristic, used in the kernel) = {SIGMA:.5f}")
print("  (SIGMA = the median distance over ALL pairs -- so about half of all pairs are")
print("   automatically treated as 'reasonably close' by the kernel, not just the true neighbours.)")

spread_ratio = (all_dists.max() - all_dists.min()) / (all_dists.mean() + 1e-12)
print(f"\n  Spread ratio (max-min)/mean = {spread_ratio:.4f}")
print("  -> if it is SMALL (<< 1): every cell looks 'similar' in the eyes of the LLM (the")
print("     representations cluster into one big blob rather than spreading out clearly per")
print("     race x religion) -- the 'nearest neighbour' is only SLIGHTLY different from 'the")
print("     furthest one', so the kernel becomes less discriminative even though the numbers look plausible on paper.")

print(f"\n--- For the 3 target cells, distance to ALL 25 other cells (not just the top 5) ---")
for cell in SPARSE_CELLS:
    i = GROUP_KEYS.index(cell)
    dists_from_i = np.delete(cos_dist_all[i], i)
    print(f"  {cell}: min={dists_from_i.min():.5f}, median={np.median(dists_from_i):.5f}, "
          f"max={dists_from_i.max():.5f}  (SIGMA={SIGMA:.5f})")

# ── G: is the JSD between neighbours ALREADY small with a RANDOM (untrained) probe? ──
print("\n--- G. How small is L_group with a RANDOM probe (0 training) -- this is the key ---")
torch.manual_seed(9999)
probe_untrained = nn.Linear(pair_emb_tensor.shape[1], 2).to(PROBE_DEVICE)
qtl_check = diag_probes[5][1]
untrained_lg = []
with torch.no_grad():
    for qkey, cell_labels in qtl_check.items():
        cells_present = list(cell_labels.keys())
        if len(cells_present) < 2:
            continue
        idx_t = torch.tensor([pair_index[(qkey, c)] for c in cells_present], device=PROBE_DEVICE)
        X = pair_emb_tensor[idx_t]
        pred = torch.softmax(probe_untrained(X), dim=-1)
        group_idx = [GROUP_INDEX[c] for c in cells_present]
        w_sub = W_IJ_full[group_idx][:, group_idx]
        total_w = w_sub.sum()
        if total_w > 0:
            jm = jsd_matrix(pred)
            untrained_lg.append(((w_sub * jm).sum() / total_w).item())

print(f"  Mean L_group, RANDOM probe (has not seen any data at all): {np.mean(untrained_lg):.4f}")
print(f"  Mean L_group, probe AFTER the 300 training epochs above  : ~0.001-0.007 (see the training log)")
print("\n  -> if this RANDOM value is ALREADY small (comparable to the value after training),")
print("     it means the geometry of the representations themselves is what makes ANY probe")
print("     (even a random one) produce similar predictions for neighbours -- NOT L_group")
print("     training successfully pulling them there. That is the most direct explanation of why")
print("     the 'final' L_group is small: it never had to work hard, they were already close from the start.")
print("     If this RANDOM value is instead LARGE (much larger than the post-training result),")
print("     it means the training did succeed in bringing them together -- but that result contradicts")
print("     the WD finding that did not improve, and would need further digging into why.")

## How to read the results

- **If `probe_L_group` (blue) sits below `probe_plain` (red), especially at small N
  (5, 10)** -- that is evidence `L_group` genuinely helps when data is scarce, in line
  with the main hypothesis of this research.
- **Compare it against `shrinkage` (green) too** -- that is the barometer for "does the
  LLM add value or not compared with pure statistics". If `probe_L_group` loses to
  `shrinkage`, that too is an honest finding (the LLM has not yet added value over
  simple statistics) -- it does not mean failure, it is a valid answer to
  RQ2 (`research_question/00_overview.md` §9).
- As N grows, all three methods should become more similar/more accurate (because the
  real data increasingly dominates and the `L_group`/shrinkage pull weakens) -- that is
  the sanity check that "the pull only applies when it is needed" (§12.2) really does
  happen.

**If the results look like the above** (`probe_L_group` ≈ `probe_plain`, both LOSING to
`shrinkage`, and no improvement even as N grows) -- **section 9 above**
(Investigations A-E) is what answers WHY, not just WHAT. Those investigation results are
the basis for deciding: does the design of `L_group`/this experiment need fixing first,
or is there already enough evidence to move to a different approach -- that decision
gets documented in turn in
`research_question/03_pivot2_group_consistency.md` once the results have been discussed.

**Next steps:**
1. Download `/kaggle/working/stage1_lgroup_pilot/`.
2. If the results look promising: extend to all 6 attribute combinations (not just
   RACExRELIG), more sparsify cells, and start trying different `lambda`/`k` values.
3. If the results are good & stable: move this training logic
   (`train_probe`, `jsd`, `build_knn_graph`, `shrinkage_predict`) into
   `scripts/` as a reusable `.py` module, so the next notebook can simply `import` it
   -- rather than copy-pasting as happened between notebooks 06 and 07.